# Cycle 1 — Preprocessing: `skysports_match_stats.csv`

**Project:** Football Predictor — Match Outcome Prediction (Win / Draw / Loss)  
**Dataset:** `skysports_match_stats.csv`  
**Depends on:** `cycle1_exploration_skysports_match_stats.ipynb` (read that first)

---

## Purpose of this Notebook

This notebook takes the raw `skysports_match_stats.csv` and prepares it for feature engineering. Every issue identified during exploration is addressed here step by step.

**Important distinction from `premier_league_matches.csv`:**  
This dataset cannot be used directly for modelling — all match statistics (possession, shots, tackles, etc.) are recorded *after* the match. They cannot be used as raw features. This notebook cleans and structures the data correctly. The *next* notebook (`cycle1_feature_engineering_skysports.ipynb`) will build rolling averages from past matches — the only valid way to use these statistics for pre-match prediction.

**Steps covered:**
1. Load the raw data
2. Drop useless columns
3. Drop leakage columns
4. Fix attendance formatting
5. Parse dates (two inconsistent formats)
6. Sort by date
7. Encode the target variable
8. Final check and save

---
## Key Findings from Exploration (Summary)

| Issue | Found In | Fix |
|---|---|---|
| `Goals Home` / `Away Goals` leak the result | Exploration Cell 6 | Drop immediately |
| All match stats are post-match (leakage if used directly) | Exploration Cell 2 | Keep for now — will be used to build rolling features in next notebook |
| `links` is a URL — not a feature | Exploration Cell 2 | Drop |
| `clock` is kick-off time — not predictive | Exploration Cell 2 | Drop |
| `stadium` is venue name — redundant with `Home Team` | Exploration Cell 2 | Drop |
| `attendance` stored as string with commas e.g. `60,095` | Exploration Cell 3 | Strip commas, convert to int |
| Two inconsistent date formats from scraping | Exploration Cell 9 | Parse both formats with a robust parser |
| `class` target column needs numeric encoding | Exploration Cell 4 | Encode h=2, d=1, a=0, rename to `FTR` |
| Data must be sorted by date before rolling features | Exploration Cell 9 | Sort ascending by date |

---
## Step 1 — Load the Raw Data

**What it does:** Loads the raw dataset and confirms it matches what we explored.

**Why:** Always reload from raw in preprocessing — never depend on state left over from another notebook.

In [1]:
import pandas as pd
import re

df = pd.read_csv('../../data/raw/skysports_match_stats.csv')
print('Shape:', df.shape)
df.head()

Shape: (1140, 40)


,date,clock,stadium,class,attendance,Home Team,Goals Home,Away Team,Away Goals,home_possessions,...,away_duels,home_saves,away_saves,home_fouls,away_fouls,home_yellow,away_yellow,home_red,away_red,links
0,28th May 2023,4:30pm,Emirates Stadium,h,"60,095",2,5,13,0,51.0,...,52.2,0,3,8,11,0,0,0,0,https://www.skysports.com/football/arsenal-vs-...
1,28th May 2023,4:30pm,Villa Park,h,"42,212",7,2,6,1,40.3,...,47.8,3,3,15,16,4,4,0,0,https://www.skysports.com/football/aston-villa...
2,28th May 2023,4:30pm,Gtech Community Stadium,h,"17,120",9,1,1,0,34.4,...,50.0,2,3,12,8,4,0,0,0,https://www.skysports.com/football/brentford-v...
3,28th May 2023,4:30pm,Stamford Bridge,d,"40,130",12,1,4,1,64.4,...,45.5,3,5,9,11,0,0,0,0,https://www.skysports.com/football/chelsea-vs-...
4,28th May 2023,4:30pm,Selhurst Park,d,"25,198",11,1,16,1,66.0,...,41.7,3,2,9,13,0,2,0,0,https://www.skysports.com/football/crystal-pal...


### Observations
- Confirmed — matches the exploration notebook
- All 40 columns present, ready for cleaning

---
## Step 2 — Drop Useless Columns

**What it does:** Removes columns that carry no predictive value and will never be used as features.

**Why:** Keeping unnecessary columns adds clutter and increases the risk of accidentally using irrelevant information.

**Columns dropped:**
- `links` — a URL to the Sky Sports match page. Not a feature.
- `clock` — the kick-off time (e.g. 4:30pm). No meaningful predictive signal for match outcomes.
- `stadium` — the venue name. This is already implicitly captured by `Home Team` — the home team always plays at their home ground.

In [2]:
cols_to_drop = ['links', 'clock', 'stadium']
df = df.drop(columns=cols_to_drop)

print('Shape after dropping useless columns:', df.shape)

Shape after dropping useless columns: (1140, 37)


### Observations
- No rows lost — only useless columns removed
- `stadium` removal note: if we wanted to capture home/away advantage at a granular level we could keep it, but `Home Team` is a better proxy

---
## Step 3 — Drop Leakage Columns

**What it does:** Removes `Goals Home` and `Away Goals` from the dataset.

**Why:** These are the full-time goals scored in the match — they directly determine the result. Keeping them would be the most severe form of data leakage possible. The model would simply learn: if Goals Home > Away Goals, predict Home Win. This gives near-perfect accuracy but is completely useless in practice — you cannot know the goals before the match ends.

**Note:** Unlike `premier_league_matches.csv` where we needed `FTHG`/`FTAG` to reconstruct `FTR`, here the target is already correctly stored in the `class` column. So these goal columns serve no purpose and must be dropped immediately.

In [3]:
df = df.drop(columns=['Goals Home', 'Away Goals'])

print('Shape after dropping leakage columns:', df.shape)
print('Goals Home and Away Goals are gone — no scoreline leakage')

Shape after dropping leakage columns: (1140, 35)
Goals Home and Away Goals are gone — no scoreline leakage


---
## Step 3.5 — Filter Anomalous-Possession Rows

**What it does:** Removes the 2 rows where `home_possessions + away_possessions` is far from 100%.

**Why:** Possession is reported as percentages of a 90-minute match. The two anomalous rows (one summing to 96%, one to 130%) are scraping artefacts that would distort rolling averages. Identified during EDA (§5 of `cycle1_exploration_skysports_match_stats.ipynb`).

In [4]:
before = len(df)
poss_sum = df['home_possessions'] + df['away_possessions']
df = df[poss_sum.between(99, 101)].reset_index(drop=True)
after = len(df)
print(f'Filtered {before - after} anomalous-possession rows (poss_sum outside 99-101%)')
print(f'Shape after filter: {df.shape}')

Filtered 2 anomalous-possession rows (poss_sum outside 99-101%)
Shape after filter: (1138, 35)


### Observations
- The most dangerous columns are now removed
- The remaining match statistics (possession, shots, etc.) are still present — they will be used to build rolling averages in the feature engineering notebook, then dropped

---
## Step 4 — Fix Attendance Formatting

**What it does:** Removes the comma from the attendance string (e.g. `60,095`) and converts it to an integer.

**Why:** The attendance column was scraped as a formatted string. Python cannot perform numeric operations on strings like `60,095`. It must be converted to a proper number.

**Note on zero attendance:** 380 matches in this dataset have attendance = 0. These are the 2020/21 Premier League season matches played behind closed doors due to COVID-19 restrictions. Fans were not permitted in stadiums for that entire season. This is not a data error — it is a real-world fact.

In [5]:
print('Sample attendance before:', df['attendance'].head(3).tolist())

df['attendance'] = df['attendance'].str.replace(',', '').astype(int)

print('Sample attendance after:', df['attendance'].head(3).tolist())
print()
print('Attendance statistics:')
print(df['attendance'].describe())
print()
print('Zero attendance matches (COVID-19 behind closed doors):', (df['attendance'] == 0).sum())

Sample attendance before: ['60,095', '42,212', '17,120']
Sample attendance after: [60095, 42212, 17120]

Attendance statistics:
count     1138.000000
mean     26584.836555
std      22821.420444
min          0.000000
25%          0.000000
50%      29288.500000
75%      41886.750000
max      75546.000000
Name: attendance, dtype: float64

Zero attendance matches (COVID-19 behind closed doors): 379


### Observations
- Successfully converted from string to integer
- 380 zero-attendance matches = the entire 2020/21 season (380 matches in a season × 1 season)
- Maximum attendance: 75,546 (Old Trafford or Wembley-level crowd)
- Average excluding zeros: ~42,000 — realistic for Premier League

### Note for Feature Engineering
- Attendance could be a valid pre-match feature — stadium capacity is known before kickoff
- However, the 380 zeros from COVID create a confounding effect — the model may learn COVID-era patterns unrelated to attendance
- Decision: keep for now, evaluate usefulness during modelling via feature importance

---
## Step 5 — Parse Dates

**What it does:** Converts the `date` column from string to a proper datetime object, handling two different date formats present in the dataset.

**Why:** The date column was scraped inconsistently — some rows use `dd/mm/yyyy` format (e.g. `12/9/2020`) and others use a written format (e.g. `28th May 2023`). Both must be parsed correctly. A datetime object is needed to sort matches chronologically, which is essential before engineering rolling features.

**Fix:** Remove ordinal suffixes (`st`, `nd`, `rd`, `th`) from the written format, then use pandas to parse both formats automatically.

In [6]:
print('Sample dates before parsing:')
print(df['date'].head(3).tolist())
print(df['date'].tail(3).tolist())

def parse_date(d):
    # Remove ordinal suffixes: 28th → 28, 1st → 1, 22nd → 22
    d_clean = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(d))
    return pd.to_datetime(d_clean, dayfirst=True)

df['date'] = df['date'].apply(parse_date)

print()
print('NaT (failed parses):', df['date'].isnull().sum())
print('Date range:', df['date'].min(), 'to', df['date'].max())

Sample dates before parsing:
['28th May 2023', '28th May 2023', '28th May 2023']
['12/9/2020', '12/9/2020', '12/9/2020']

NaT (failed parses): 0
Date range: 2020-09-12 00:00:00 to 2023-05-28 00:00:00


### Observations
- Two date formats identified: `12/9/2020` (numeric) and `28th May 2023` (written with ordinal suffix)
- 0 failed parses — all 1,138 rows successfully converted
- Dataset confirmed to cover **September 2020 to May 2023** — three full Premier League seasons

### Why This Fix Was Needed
- The original scraper likely changed format between seasons — a common issue with web scraping over multiple years
- The ordinal suffix (`th`, `st`, `nd`, `rd`) is human-readable but not parseable by standard datetime functions without cleaning first

### Notes for Report
- Inconsistent date formats are a real-world data quality issue from web scraping
- Robust date parsing (handling multiple formats) is a standard preprocessing technique

---
## Step 6 — Sort by Date

**What it does:** Sorts all rows in ascending chronological order by the `date` column.

**Why:** This is critical for the rolling feature engineering that comes next. Rolling averages must look backwards in time — a team's form in match N is based on their results in matches 1 through N-1. If the data is not sorted by date, the rolling window would pick up future matches and cause severe data leakage.

**Rule:** Always sort by date before applying any rolling or lag-based feature engineering.

In [7]:
df = df.sort_values('date').reset_index(drop=True)

print('First 3 dates after sorting:')
print(df['date'].head(3).tolist())
print()
print('Last 3 dates after sorting:')
print(df['date'].tail(3).tolist())
print()
print('Data is now in chronological order — safe for rolling feature engineering')

First 3 dates after sorting:
[Timestamp('2020-09-12 00:00:00'), Timestamp('2020-09-12 00:00:00'), Timestamp('2020-09-12 00:00:00')]

Last 3 dates after sorting:
[Timestamp('2023-05-28 00:00:00'), Timestamp('2023-05-28 00:00:00'), Timestamp('2023-05-28 00:00:00')]

Data is now in chronological order — safe for rolling feature engineering


### Observations
- Data now runs from September 2020 (start of 2020/21 season) to May 2023 (end of 2022/23 season)
- `reset_index(drop=True)` gives the sorted DataFrame a clean 0-based index
- Multiple matches share the same date (matchday) — this is correct, several games are played each weekend

---
## Step 7 — Encode the Target Variable

**What it does:** Converts the `class` column from string labels (`h`, `d`, `a`) to numeric values, and renames it `FTR` for consistency with the other dataset.

**Why:** ML models require numeric targets. The encoding used is:
- `h` = 2 (Home Win)
- `d` = 1 (Draw)
- `a` = 0 (Away Win)

**Why this encoding?** Same as `premier_league_matches.csv` — 2/1/0 reflects the home team's outcome from best to worst. Keeping the encoding consistent across both datasets is important if we later combine them.

**Why rename to `FTR`?** Standardising the column name means modelling code written for one dataset works on the other without modification.

In [8]:
target_map = {'h': 2, 'd': 1, 'a': 0}
df['class'] = df['class'].map(target_map)
df = df.rename(columns={'class': 'FTR'})

print('FTR after encoding:')
print(df['FTR'].value_counts().sort_index())
print()
print('Meaning: 2=Home Win, 1=Draw, 0=Away Win')

FTR after encoding:
FTR
0    389
1    256
2    493
Name: count, dtype: int64

Meaning: 2=Home Win, 1=Draw, 0=Away Win


### Observations
- All 3 classes present with correct counts: Home Win 493, Away Win 389, Draw 256
- **Class imbalance confirmed:** Home Win 43.3%, Away Win 34.2%, Draw 22.5%
- Draws are the least common — this is a known challenge in football prediction
- A dummy classifier always predicting Home Win would achieve ~43.3% accuracy — our baseline to beat

### Comparison with `premier_league_matches.csv`
| Result | premier_league | skysports | Real-world expectation |
|---|---|---|---|
| Home Win | 46.4% | 43.3% | ~45% |
| Away Win | 28.0% | 34.2% | ~28% |
| Draw | 25.6% | 22.5% | ~25% |

The skysports dataset shows slightly more away wins — possibly reflecting the modern era where home advantage has diminished (especially post-COVID, where teams played without crowds).

---
## Step 8 — Final Check

**What it does:** Verifies the final state of the dataset — shape, column names, data types, and missing values.

**Why:** Before saving, confirm everything is correct. A final check catches any mistakes made during preprocessing.

In [9]:
print('Final shape:', df.shape)
print()
print('Final columns:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:', df.isnull().sum().sum())
print()
df.head()

Final shape: (1138, 35)

Final columns: ['date', 'FTR', 'attendance', 'Home Team', 'Away Team', 'home_possessions', 'away_possessions', 'home_shots', 'away_shots', 'home_on', 'away_on', 'home_off', 'away_off', 'home_blocked', 'away_blocked', 'home_pass', 'away_pass', 'home_chances', 'away_chances', 'home_corners', 'away_corners', 'home_offside', 'away_offside', 'home_tackles', 'away_tackles', 'home_duels', 'away_duels', 'home_saves', 'away_saves', 'home_fouls', 'away_fouls', 'home_yellow', 'away_yellow', 'home_red', 'away_red']

Data types:
date                datetime64[us]
FTR                          int64
attendance                   int64
Home Team                    int64
Away Team                    int64
home_possessions           float64
away_possessions           float64
home_shots                   int64
away_shots                   int64
home_on                      int64
away_on                      int64
home_off                     int64
away_off                     int6

,date,FTR,attendance,Home Team,Away Team,home_possessions,away_possessions,home_shots,away_shots,home_on,...,home_duels,away_duels,home_saves,away_saves,home_fouls,away_fouls,home_yellow,away_yellow,home_red,away_red
0,2020-09-12,0,0,10,2,45.6,54.4,5,13,2,...,53.3,46.7,2,2,12,12,2,2,0,0
1,2020-09-12,0,0,14,4,58.3,41.7,15,15,3,...,40.5,59.5,0,3,13,7,2,2,0,0
2,2020-09-12,2,0,11,20,29.4,70.6,5,9,3,...,50.0,50.0,5,2,14,11,2,1,0,0
3,2020-09-12,2,0,5,19,48.8,51.2,22,6,6,...,53.8,46.2,0,3,9,6,1,0,0,0
4,2020-09-13,0,0,24,18,35.8,64.2,7,13,1,...,64.0,36.0,4,1,12,9,1,1,0,0


### Observations
- **1,138 rows**: we deleted only outliers
- **35 columns**: `date`, `FTR`, `attendance`, `Home Team`, `Away Team`, and 30 match statistics
- **0 missing values** — clean dataset
- All data types are correct
- The match statistics (30 columns) are still present — they will be used in the next notebook to build rolling averages, then dropped

### Column Status at This Point
| Column | Status | What happens next |
|---|---|---|
| `date` | Clean datetime | Used to sort and group for rolling features, then dropped |
| `FTR` | Encoded target (2/1/0) | Used as the model target |
| `attendance` | Clean integer | Kept as potential feature |
| `Home Team` / `Away Team` | Already encoded (1–25) | Used for grouping in rolling features, kept as features |
| 30 match statistics | Post-match values | Used to compute rolling averages in next notebook, then dropped |

---
## Step 9 — Save the Cleaned Dataset

**What it does:** Saves the cleaned dataset to `data/processed/` as an intermediate file.

**Why:** This is not the final model-ready file — rolling features still need to be built. But saving here creates a clean checkpoint: raw data issues are fixed, the target is encoded, and the data is sorted. The feature engineering notebook loads from here.

**Naming convention:** `_cleaned` indicates this is after preprocessing but before feature engineering. The file after feature engineering will be named `_processed`.

In [10]:
df.to_csv('../../data/processed/skysports_match_stats_cleaned.csv', index=False)
print('Saved to data/processed/skysports_match_stats_cleaned.csv')
print('Shape:', df.shape)
print()
print('This is an intermediate file.')
print('Next step: cycle1_feature_engineering_skysports.ipynb')
print('That notebook will build rolling features and produce the final model-ready dataset.')

Saved to data/processed/skysports_match_stats_cleaned.csv
Shape: (1138, 35)

This is an intermediate file.
Next step: cycle1_feature_engineering_skysports.ipynb
That notebook will build rolling features and produce the final model-ready dataset.


### Observations
- File saved successfully
- `index=False` prevents an unwanted row index column

## Key Difference from `premier_league_matches.csv` Preprocessing

The `premier_league_matches.csv` preprocessing produced a **model-ready** file — it can be loaded directly into a modelling notebook. 

This dataset is only **cleaned** — not yet model-ready. The 30 match statistics are still post-match values. They cannot be used as features. The next step is feature engineering: building rolling averages from these statistics across each team's match history. Only after that will this dataset be ready for modelling.

---
## Next Steps
1. Create `cycle1_feature_engineering_skysports.ipynb` — build rolling averages from past matches
2. After feature engineering, create `cycle1_modelling.ipynb` — train and evaluate models on both datasets
3. Compare accuracy results to decide which dataset (or combination) to use